In [1]:
import json, os, shutil, random, zipfile
from pathlib import Path
from PIL import Image as PIL_Image
import numpy as np
import xml.etree.ElementTree as ET

# ── Paths ────────────────────────────────────────────────────────────
TACO_DATA_DIR         = Path("datasets/taco/data")
TACO_ANN_FILE         = TACO_DATA_DIR / "annotations.json"
FLS_ROOT              = Path("datasets/marine_debris/md_fls_dataset/data")
FLS_SEG_DIR           = FLS_ROOT / "watertank-segmentation"
FLS_TURNTABLE_DIR     = FLS_ROOT / "turntable-cropped"
FLS_WATERTANK_CROPPED = FLS_ROOT / "watertank-cropped"
FLS_QUARRY_DIR        = FLS_ROOT / "quarry-fullsize"
TRASHNET_ZIP          = Path("datasets/trashnet/data/dataset-resized.zip")
TRASHNET_EXTRACT      = Path("datasets/trashnet/data/dataset-resized")
SAVE_DIR              = Path("Dataset")

# ── Splits ───────────────────────────────────────────────────────────
TRAIN_RATIO  = 0.80
VAL_RATIO    = 0.10
RANDOM_SEED  = 42
USE_GRAYSCALE_FOR_TACO = False

# ── Shared taxonomy ──────────────────────────────────────────────────
CLASS_NAMES = ["plastic", "metal", "glass", "rubber", "paper", "organic", "debris"]
PLASTIC, METAL, GLASS, RUBBER, PAPER, ORGANIC, DEBRIS = range(7)

# ── FLS XML class map (skip Wall, rotating-platform) ─────────────────
FLS_XML_CLASS_MAP = {
    "can"              : METAL,
    "chain"            : METAL,
    "hook"             : METAL,
    "propeller"        : METAL,
    "valve"            : METAL,
    "metal-bottle"     : METAL,
    "metal-box"        : METAL,
    "wrench"           : METAL,
    "bottle"           : PLASTIC,
    "plastic-bottle"   : PLASTIC,
    "plastic-bidon"    : PLASTIC,
    "plastic-pipe"     : PLASTIC,
    "shampoo-bottle"   : PLASTIC,
    "standing-bottle"  : PLASTIC,
    "drink-sachet"     : PLASTIC,
    "brown-glass-bottle": GLASS,
    "glass-bottle"     : GLASS,
    "glass-jar"        : GLASS,
    "potion-glass-bottle": GLASS,
    "large-tire"       : RUBBER,
    "small-tire"       : RUBBER,
    "tire"             : RUBBER,
    "drink-carton"     : PAPER,
    "wall"             : None,   # skip
    "rotating-platform": None,   # skip
}

# ── Turntable / watertank cropped folder → class ─────────────────────
FLS_FOLDER_MAP = {
    "brown-glass-bottle"  : GLASS,
    "glass-bottle"        : GLASS,
    "glass-jar"           : GLASS,
    "potion-glass-bottle" : GLASS,
    "can"                 : METAL,
    "metal-bottle"        : METAL,
    "metal-box"           : METAL,
    "wrench"              : METAL,
    "valve"               : METAL,
    "chain"               : METAL,
    "hook"                : METAL,
    "propeller"           : METAL,
    "drink-carton"        : PAPER,
    "drink-sachet"        : PAPER,
    "large-tire"          : RUBBER,
    "small-tire"          : RUBBER,
    "tire"                : RUBBER,
    "plastic-bottle"      : PLASTIC,
    "plastic-bidon"       : PLASTIC,
    "plastic-pipe"        : PLASTIC,
    "plastic-propeller"   : PLASTIC,
    "shampoo-bottle"      : PLASTIC,
    "standing-bottle"     : PLASTIC,
    "bottle"              : PLASTIC,
    "rotating-platform"   : None,   # skip
}

# ── TrashNet folder → class ───────────────────────────────────────────
TRASHNET_MAP = {
    "plastic"   : PLASTIC,
    "metal"     : METAL,
    "glass"     : GLASS,
    "paper"     : PAPER,
    "cardboard" : PAPER,
    "trash"     : DEBRIS,
}

# ── TACO keyword → class ─────────────────────────────────────────────
TACO_KEYWORD_MAP = {
    "plastic": PLASTIC, "bottle": PLASTIC, "bag": PLASTIC,
    "film": PLASTIC, "cup": PLASTIC, "straw": PLASTIC,
    "lid": PLASTIC, "cap": PLASTIC, "wrapper": PLASTIC,
    "styrofoam": PLASTIC, "foam": PLASTIC, "cigarette": PLASTIC,
    "can": METAL, "metal": METAL, "aluminium": METAL,
    "foil": METAL, "tin": METAL, "wire": METAL,
    "glass": GLASS, "jar": GLASS,
    "rubber": RUBBER, "tyre": RUBBER, "tire": RUBBER,
    "glove": RUBBER, "boot": RUBBER,
    "paper": PAPER, "cardboard": PAPER, "carton": PAPER,
    "food": ORGANIC, "wood": ORGANIC, "rope": ORGANIC,
    "textile": ORGANIC, "cloth": ORGANIC,
}

def taco_cat_to_class(name):
    n = name.lower().replace(" ", "_")
    for kw, cls in TACO_KEYWORD_MAP.items():
        if kw in n:
            return cls
    return DEBRIS

In [2]:
# ── Helpers ───────────────────────────────────────────────────────────
def make_dirs():
    for split in ("train", "val", "test"):
        (SAVE_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (SAVE_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

def assign_split(idx, total):
    r = idx / total
    if r < TRAIN_RATIO: return "train"
    elif r < TRAIN_RATIO + VAL_RATIO: return "val"
    return "test"

def write_yolo_label(path, boxes):
    with open(path, "w") as f:
        for b in boxes:
            f.write(f"{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n")

def coco_bbox_to_yolo(bbox, W, H):
    x, y, w, h = bbox
    return (x + w/2)/W, (y + h/2)/H, w/W, h/H

def clamp_box(cx, cy, w, h):
    return (max(0.0, min(1.0, cx)), max(0.0, min(1.0, cy)),
            max(0.001, min(1.0, w)), max(0.001, min(1.0, h)))

def _print_split_counts(counters):
    print(f"   Running totals → train={counters['train']}  "
          f"val={counters['val']}  test={counters['test']}", flush=True)

In [3]:
def process_taco(counters):
    if not TACO_ANN_FILE.exists():
        print(f"⚠️  TACO not found — skipping.", flush=True); return

    print("\n── TACO ──────────────────────────────────────────────────────", flush=True)
    with open(TACO_ANN_FILE) as f:
        ann = json.load(f)

    cat_map  = {c["id"]: taco_cat_to_class(c["name"]) for c in ann["categories"]}
    img_anns = {}
    for a in ann["annotations"]:
        img_anns.setdefault(a["image_id"], []).append(a)

    images = ann["images"][:]
    random.shuffle(images)

    ok = skip = 0
    for idx, img_info in enumerate(images):
        split    = assign_split(idx, len(images))
        img_name = Path(img_info["file_name"])
        src_path = TACO_DATA_DIR / img_name
        if not src_path.exists():
            skip += 1; continue
        try:
            with PIL_Image.open(src_path) as im:
                W, H = im.size
                dst_img = SAVE_DIR / "images" / split / f"taco_{img_name.name}"
                if USE_GRAYSCALE_FOR_TACO:
                    im.convert("L").convert("RGB").save(dst_img, quality=92)
                else:
                    shutil.copy2(src_path, dst_img)
        except Exception:
            skip += 1; continue

        boxes = []
        for a in img_anns.get(img_info["id"], []):
            cx, cy, w, h = clamp_box(*coco_bbox_to_yolo(a["bbox"], W, H))
            boxes.append((cat_map.get(a["category_id"], DEBRIS), cx, cy, w, h))

        write_yolo_label(SAVE_DIR / "labels" / split / f"taco_{img_name.stem}.txt", boxes)
        counters[split] += 1; ok += 1

    print(f"   ✅ {ok} images  ({skip} skipped)", flush=True)
    _print_split_counts(counters)

In [4]:
def process_fls_segmentation(counters):
    if not FLS_SEG_DIR.exists():
        print(f"⚠️  FLS segmentation not found — skipping.", flush=True); return

    img_dir = FLS_SEG_DIR / "Images"
    box_dir = FLS_SEG_DIR / "BoxAnnotations"
    msk_dir = FLS_SEG_DIR / "Masks"

    print("\n── Marine FLS: Watertank Segmentation ───────────────────────", flush=True)
    img_paths = sorted(img_dir.glob("*.png")) + sorted(img_dir.glob("*.jpg"))
    random.shuffle(img_paths)
    ok = skip = 0

    for idx, img_path in enumerate(img_paths):
        split = assign_split(idx, len(img_paths))
        stem  = img_path.stem
        boxes = []

        # ── Parse XML bounding boxes ──────────────────────────────────
        xml_path = box_dir / (stem + ".xml")
        if xml_path.exists():
            try:
                tree = ET.parse(xml_path)
                root = tree.getroot()
                W = int(root.find("size/width").text)
                H = int(root.find("size/height").text)
                for obj in root.findall("object"):
                    name   = obj.find("name").text.strip().lower()
                    cls_id = FLS_XML_CLASS_MAP.get(name)
                    if cls_id is None:
                        continue   # skip Wall etc.
                    bb  = obj.find("bndbox")
                    x   = float(bb.find("x").text)
                    y   = float(bb.find("y").text)
                    w   = float(bb.find("w").text)
                    h   = float(bb.find("h").text)
                    cx, cy, bw, bh = clamp_box(*coco_bbox_to_yolo([x, y, w, h], W, H))
                    boxes.append((cls_id, cx, cy, bw, bh))
            except Exception:
                pass   # fall through to mask fallback

        # ── Mask fallback if no boxes parsed ─────────────────────────
        if not boxes:
            mask_path = msk_dir / (stem + ".png")
            if mask_path.exists():
                try:
                    with PIL_Image.open(img_path) as im:
                        W, H = im.size
                    mask_arr = np.array(PIL_Image.open(mask_path).convert("RGB"))
                    for colour in np.unique(mask_arr.reshape(-1, 3), axis=0):
                        if tuple(colour) in ((0,0,0), (255,255,255)):
                            continue
                        m    = np.all(mask_arr == colour, axis=2)
                        rows = np.where(m.any(axis=1))[0]
                        cols = np.where(m.any(axis=0))[0]
                        if not len(rows) or not len(cols):
                            continue
                        cx  = ((cols[0]+cols[-1])/2)/W
                        cy  = ((rows[0]+rows[-1])/2)/H
                        bw  = (cols[-1]-cols[0])/W
                        bh  = (rows[-1]-rows[0])/H
                        if bw > 0.01 and bh > 0.01:
                            boxes.append(clamp_box(cx, cy, bw, bh))
                            boxes[-1] = (DEBRIS,) + boxes[-1]
                except Exception:
                    pass

        if not boxes:
            skip += 1; continue

        shutil.copy2(img_path, SAVE_DIR / "images" / split / f"flsseg_{stem}.png")
        write_yolo_label(SAVE_DIR / "labels" / split / f"flsseg_{stem}.txt", boxes)
        counters[split] += 1; ok += 1

    print(f"   ✅ {ok} images  ({skip} skipped)", flush=True)
    _print_split_counts(counters)

In [5]:
def _process_folder_dataset(counters, src_dir, folder_map, prefix):
    """Generic processor for class-named folder datasets (turntable, watertank-cropped)."""
    samples = []
    for class_dir in sorted(src_dir.iterdir()):
        if not class_dir.is_dir(): continue
        cls_id = folder_map.get(class_dir.name.lower())
        if cls_id is None: continue   # skip rotating-platform etc.
        for img_path in list(class_dir.glob("*.png")) + list(class_dir.glob("*.jpg")):
            samples.append((img_path, cls_id, class_dir.name.lower()))

    random.shuffle(samples)
    ok = skip = 0
    for idx, (img_path, cls_id, cls_name) in enumerate(samples):
        split    = assign_split(idx, len(samples))
        stem     = f"{cls_name}_{img_path.stem}"
        dst_img  = SAVE_DIR / "images" / split / f"{prefix}_{stem}.png"
        lbl_path = SAVE_DIR / "labels" / split / f"{prefix}_{stem}.txt"
        try:
            shutil.copy2(img_path, dst_img)
        except Exception:
            skip += 1; continue
        write_yolo_label(lbl_path, [(cls_id, 0.5, 0.5, 0.95, 0.95)])
        counters[split] += 1; ok += 1

    print(f"   ✅ {ok} images  ({skip} skipped)", flush=True)
    _print_split_counts(counters)

def process_fls_turntable(counters):
    if not FLS_TURNTABLE_DIR.exists():
        print("⚠️  Turntable dir not found — skipping.", flush=True); return
    print("\n── Marine FLS: Turntable Cropped ────────────────────────────", flush=True)
    _process_folder_dataset(counters, FLS_TURNTABLE_DIR, FLS_FOLDER_MAP, "flstt")

def process_fls_watertank_cropped(counters):
    if not FLS_WATERTANK_CROPPED.exists():
        print("⚠️  Watertank-cropped dir not found — skipping.", flush=True); return
    print("\n── Marine FLS: Watertank Cropped ────────────────────────────", flush=True)
    _process_folder_dataset(counters, FLS_WATERTANK_CROPPED, FLS_FOLDER_MAP, "flswt")

In [6]:
def process_fls_quarry(counters):
    if not FLS_QUARRY_DIR.exists():
        print("⚠️  Quarry dir not found — skipping.", flush=True); return

    print("\n── Marine FLS: Quarry Fullsize ──────────────────────────────", flush=True)
    # Quarry has timestamped subdirs with raw sonar frames, no object labels
    # Included as weak DEBRIS supervision
    img_paths = list(FLS_QUARRY_DIR.rglob("*.png")) + list(FLS_QUARRY_DIR.rglob("*.jpg"))
    random.shuffle(img_paths)
    ok = skip = 0
    for idx, img_path in enumerate(img_paths):
        split    = assign_split(idx, len(img_paths))
        stem     = img_path.stem
        dst_img  = SAVE_DIR / "images" / split / f"flsqr_{stem}.png"
        lbl_path = SAVE_DIR / "labels" / split / f"flsqr_{stem}.txt"
        try:
            shutil.copy2(img_path, dst_img)
        except Exception:
            skip += 1; continue
        write_yolo_label(lbl_path, [(DEBRIS, 0.5, 0.5, 0.95, 0.95)])
        counters[split] += 1; ok += 1

    print(f"   ✅ {ok} images  ({skip} skipped)", flush=True)
    _print_split_counts(counters)

In [7]:
def process_trashnet(counters):
    # Extract zip if not already done
    if not TRASHNET_EXTRACT.exists():
        print("   Extracting TrashNet zip...", flush=True)
        with zipfile.ZipFile(TRASHNET_ZIP) as z:
            z.extractall(TRASHNET_ZIP.parent)

    print("\n── TrashNet ──────────────────────────────────────────────────", flush=True)
    _process_folder_dataset(counters, TRASHNET_EXTRACT, TRASHNET_MAP, "trashnet")

In [8]:
def write_yaml():
    abs_save = SAVE_DIR.resolve()
    content = f"""path: {abs_save}
train: images/train
val:   images/val
test:  images/test

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""
    yaml_path = SAVE_DIR / "data.yaml"
    with open(yaml_path, "w") as f:
        f.write(content)
    print(f"\n📄 data.yaml → {yaml_path}", flush=True)
    return yaml_path

def verify_and_report(counters, yaml_path):
    from collections import Counter
    print("\n═══════════════════════════════════════════════════════", flush=True)
    print("  Final Dataset Report", flush=True)
    print("═══════════════════════════════════════════════════════", flush=True)
    grand_total = 0
    for split in ("train", "val", "test"):
        imgs    = list((SAVE_DIR / "images" / split).glob("*"))
        lbls    = list((SAVE_DIR / "labels" / split).glob("*.txt"))
        matched = {p.stem for p in imgs} & {p.stem for p in lbls}
        print(f"  {split:5s}  images={len(imgs):5d}  labels={len(lbls):5d}  matched={len(matched):5d}", flush=True)
        grand_total += len(matched)

    class_counts = Counter()
    for split in ("train", "val", "test"):
        for lbl_file in (SAVE_DIR / "labels" / split).glob("*.txt"):
            for line in open(lbl_file):
                parts = line.strip().split()
                if parts: class_counts[int(parts[0])] += 1

    print(f"\n  Total: {grand_total}", flush=True)
    print(f"\n  Class distribution:", flush=True)
    for i, name in enumerate(CLASS_NAMES):
        count = class_counts.get(i, 0)
        bar   = "█" * min(40, count // max(1, grand_total // 100))
        print(f"    [{i}] {name:10s}  {count:6d}  {bar}", flush=True)
    print(f"\n  YAML: {yaml_path.resolve()}", flush=True)
    print("═══════════════════════════════════════════════════════", flush=True)

In [10]:
import subprocess
result = subprocess.run(
    ["python", "download.py", "--dataset_path", "data/annotations.json"],
    cwd="datasets/taco",
    capture_output=True,
    text=True
)
print(result.stdout, flush=True)
print(result.stderr, flush=True)

Note. If for any reason the connection is broken. Just call me again and I will start where I left.
Loading: [..............................] - 0/1500
Loading: [..............................] - 1/1500
Loading: [..............................] - 2/1500
Loading: [..............................] - 3/1500
Loading: [..............................] - 4/1500
Loading: [..............................] - 5/1500
Loading: [..............................] - 6/1500
Loading: [..............................] - 7/1500
Loading: [..............................] - 8/1500
Loading: [..............................] - 9/1500
Loading: [..............................] - 10/1500
Loading: [..............................] - 11/1500
Loading: [..............................] - 12/1500
Loading: [..............................] - 13/1500
Loading: [..............................] - 14/1500
Loading: [..............................] - 15/1500
Loading: [..............................] - 16/1500
Loading: [................

In [11]:
# ── RUN ───────────────────────────────────────────────────────────────
random.seed(RANDOM_SEED)
make_dirs()
counters = {"train": 0, "val": 0, "test": 0}

process_taco(counters)
process_fls_segmentation(counters)
process_fls_turntable(counters)
process_fls_watertank_cropped(counters)
process_fls_quarry(counters)
process_trashnet(counters)

yaml_path = write_yaml()
verify_and_report(counters, yaml_path)


── TACO ──────────────────────────────────────────────────────
   ✅ 1500 images  (0 skipped)
   Running totals → train=1200  val=150  test=150

── Marine FLS: Watertank Segmentation ───────────────────────
   ✅ 1868 images  (0 skipped)
   Running totals → train=2695  val=337  test=336

── Marine FLS: Turntable Cropped ────────────────────────────
   ✅ 4756 images  (0 skipped)
   Running totals → train=6500  val=813  test=811

── Marine FLS: Watertank Cropped ────────────────────────────
   ✅ 2364 images  (0 skipped)
   Running totals → train=8392  val=1049  test=1047

── Marine FLS: Quarry Fullsize ──────────────────────────────
   ✅ 7209 images  (0 skipped)
   Running totals → train=14160  val=1770  test=1767

── TrashNet ──────────────────────────────────────────────────
   ✅ 2527 images  (0 skipped)
   Running totals → train=16182  val=2023  test=2019

📄 data.yaml → Dataset/data.yaml

═══════════════════════════════════════════════════════
  Final Dataset Report
═══════════════════

In [12]:
# Remove unmatched images that have no label
for split in ("train", "val", "test"):
    img_dir = SAVE_DIR / "images" / split
    lbl_dir = SAVE_DIR / "labels" / split
    label_stems = {p.stem for p in lbl_dir.glob("*.txt")}
    removed = 0
    for img in img_dir.glob("*"):
        if img.stem not in label_stems:
            img.unlink()
            removed += 1
    print(f"{split}: removed {removed} unmatched images", flush=True)

train: removed 0 unmatched images
val: removed 0 unmatched images
test: removed 0 unmatched images


In [2]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
model.train(
    data="Dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    hsv_s=0.4,
    hsv_v=0.4,
    mosaic=1.0,
    degrees=15,
    project="runs/microplastic",
    name="v1"
)

New https://pypi.org/project/ultralytics/8.4.30 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.21 🚀 Python-3.14.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 5806MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Dataset/data.yaml, degrees=15, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, nam

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f4cc0206b30>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
  